In [0]:
from pyspark.sql import functions as F

In [0]:
path = "/Volumes/hospital_poc/bronze/raw_data/synthetic_notes_medium_1000.json"

In [0]:
df = (
    spark.read
    .option("multiLine", "true")
    .json(path)
)

display(df.limit(10))

id,length,note,specialty,version
1,medium,"SSKM Hospital Kolkata Discharge Summary - V2 Patient Name: Baby Saanvi Age/Sex: 68/F UHID: 632376936 Admission No: ADM-20260462 Address: Patna, Bihar Admission Date: 07/03/2026 08:00 AM Discharge Date: 14/03/2026 01:00 PM Treating Consultant: Dr. Naresh Trehan (Cardiac Surgeon), Oncology Chief Complaints: Unexplained weight loss and fatigue for 8 months. History of Present Illness: Patient noticed a gradual loss of appetite. Past History: No history of major illness. Physical Examination: 2x2 cm hard lump in the upper outer quadrant of the left breast. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: CBC shows anemia. PET scan shows metastasis. Diagnosis: Metastatic Adenocarcinoma of unknown primary (ICD: C80.0). Course in Hospital: Underwent mastectomy followed by chemotherapy. Socio-Economic Status: Class IV as per modified Kuppuswamy scale. Traditional Medicine History: Used Tulsi for cough. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Regular follow-up in the oncology OPD. - Dawa time pe lena. Dr. Naresh Trehan (Cardiac Surgeon) Signature",oncology,2.0
2,medium,"Tata Memorial Hospital Mumbai Discharge Summary - V2 Patient Name: Debashish Roy Age/Sex: 85/F UHID: 247712229 Admission No: ADM-20268090 Address: Kaithal, Haryana Admission Date: 09/03/2026 09:00 AM Discharge Date: 14/03/2026 02:00 PM Treating Consultant: Dr. Anita Sharma, Cardiology Chief Complaints: Palpitations and dizziness for 3 days. History of Present Illness: Substernal pain on exertion, with diaphoresis. Patient ko fever tha. Past History: Coronary artery disease, post-stent. Physical Examination: BP: 160/102 mmHg, Heart: S1/S2 normal, no murmur. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: ECG: ST elevation; Trop-I: Positive; Echo: EF 45%; Angiogram: 52% blockage in LAD. Diagnosis: Atrial Fibrillation (ICD: I48.91). Course in Hospital: Antiarrhythmics given, rhythm stabilized. Socio-Economic Status: Class III as per modified Kuppuswamy scale. Traditional Medicine History: Used Tulsi for cough. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Meds: Tab. Aspirin 75mg OD, Tab. Atorvastatin 40mg HS. - pura course complete karna. Dr. Arvind Subramanian (Cardiologist) Signature",cardiology,2.0
3,medium,"Apollo Hospital Chennai Discharge Summary - V2 Patient Name: Jayesh Patel Age/Sex: 85/F UHID: 805408206 Admission No: ADM-20267292 Address: Kaithal, Haryana Admission Date: 03/03/2026 09:00 AM Discharge Date: 10/03/2026 03:00 PM Treating Consultant: Dr. Rajesh Kumar, General_medicine Chief Complaints: Abdominal pain for 22 days. History of Present Illness: Productive cough, no hemoptysis. Past History: Diabetes on insulin. Physical Examination: Temp: 102°F; Chest: Crepitations. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: Endoscopy: Gastritis. Diagnosis: Community-Acquired Pneumonia (ICD: J18.9). Course in Hospital: IV antibiotics. Fever subsided. Socio-Economic Status: Class IV as per modified Kuppuswamy scale. Traditional Medicine History: Used Haldi milk for pain. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Diet: Bland; Follow-up: 1 week. Dr. Vikram Singh Signature",general_medicine,2.0
4,medium,"AIIMS Delhi Discharge Summary - V2 Patient Name: Imran Khan Age/Sex: 35/M UHID: 926990054 Admission No: ADM-20263390 Address: Darbhanga, Bihar Admission Date: 09/03/2026 11:00 AM Discharge Date: 13/03/2026 02:00 PM Treating Consultant: Dr. Arvind Subramanian (Cardiologist), Pediatrics Chief Complaints: Loose motions aur vomiting for 17 days. History of Present Illness: High fever, measles-like rash. Past History: Recent URI. Physical Examination: Dehydrated, abdomen soft. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, L

In [0]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- length: string (nullable = true)
 |-- note: string (nullable = true)
 |-- specialty: string (nullable = true)
 |-- version: string (nullable = true)



In [0]:
print("total records: ",df.count())

total records:  1000


In [0]:
duplicate_count=df.count()-df.dropDuplicates().count()
print("duplicate rows: ",duplicate_count)

duplicate rows:  0


In [0]:
duplicate_ids=(
    df.groupBy("id")
    .count()
    .filter(F.col("Count")>1)
)
display(duplicate_ids)

id,count


In [0]:
null_counts = df.select([
    F.count(
        F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c)
    ).alias(c)
    for c in df.columns
])

display(null_counts)

id,length,note,specialty,version
0,0,0,0,0


In [0]:
display(
    df.groupBy("specialty")
    .count()
    .orderBy(F.desc("Count"))
)

specialty,count
oncology,208
general_medicine,206
neurology,202
cardiology,193
pediatrics,191


In [0]:
display(
    df.groupBy("version")
    .count()
)

version,count
2.0,1000


In [0]:
display(
    df.groupBy("length")
    .count()
)

length,count
medium,1000


In [0]:
df_with_length=df.withColumn(
    "note_length",
    F.length("note")
)
display(
    df_with_length.select("id","specialty","note_length")
    .orderBy(F.desc("note_length"))
)

id,specialty,note_length
805,cardiology,1251
967,cardiology,1248
817,cardiology,1248
105,cardiology,1247
24,cardiology,1242
971,cardiology,1241
779,cardiology,1229
752,cardiology,1224
502,cardiology,1223
644,cardiology,1221


In [0]:
display(
    df_with_length.select(
        F.min("note_length").alias("min_length"),
        F.max("note_length").alias("max_length"),
        F.avg("note_length").alias("avg_length")
    )
)

min_length,max_length,avg_length
934,1251,1078.188


In [0]:
display(
    df.filter(~F.col("note").contains("diagnosis:"))
      .select("id", "note")
)

id,note
1,"SSKM Hospital Kolkata Discharge Summary - V2 Patient Name: Baby Saanvi Age/Sex: 68/F UHID: 632376936 Admission No: ADM-20260462 Address: Patna, Bihar Admission Date: 07/03/2026 08:00 AM Discharge Date: 14/03/2026 01:00 PM Treating Consultant: Dr. Naresh Trehan (Cardiac Surgeon), Oncology Chief Complaints: Unexplained weight loss and fatigue for 8 months. History of Present Illness: Patient noticed a gradual loss of appetite. Past History: No history of major illness. Physical Examination: 2x2 cm hard lump in the upper outer quadrant of the left breast. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: CBC shows anemia. PET scan shows metastasis. Diagnosis: Metastatic Adenocarcinoma of unknown primary (ICD: C80.0). Course in Hospital: Underwent mastectomy followed by chemotherapy. Socio-Economic Status: Class IV as per modified Kuppuswamy scale. Traditional Medicine History: Used Tulsi for cough. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Regular follow-up in the oncology OPD. - Dawa time pe lena. Dr. Naresh Trehan (Cardiac Surgeon) Signature"
2,"Tata Memorial Hospital Mumbai Discharge Summary - V2 Patient Name: Debashish Roy Age/Sex: 85/F UHID: 247712229 Admission No: ADM-20268090 Address: Kaithal, Haryana Admission Date: 09/03/2026 09:00 AM Discharge Date: 14/03/2026 02:00 PM Treating Consultant: Dr. Anita Sharma, Cardiology Chief Complaints: Palpitations and dizziness for 3 days. History of Present Illness: Substernal pain on exertion, with diaphoresis. Patient ko fever tha. Past History: Coronary artery disease, post-stent. Physical Examination: BP: 160/102 mmHg, Heart: S1/S2 normal, no murmur. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: ECG: ST elevation; Trop-I: Positive; Echo: EF 45%; Angiogram: 52% blockage in LAD. Diagnosis: Atrial Fibrillation (ICD: I48.91). Course in Hospital: Antiarrhythmics given, rhythm stabilized. Socio-Economic Status: Class III as per modified Kuppuswamy scale. Traditional Medicine History: Used Tulsi for cough. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Meds: Tab. Aspirin 75mg OD, Tab. Atorvastatin 40mg HS. - pura course complete karna. Dr. Arvind Subramanian (Cardiologist) Signature"
3,"Apollo Hospital Chennai Discharge Summary - V2 Patient Name: Jayesh Patel Age/Sex: 85/F UHID: 805408206 Admission No: ADM-20267292 Address: Kaithal, Haryana Admission Date: 03/03/2026 09:00 AM Discharge Date: 10/03/2026 03:00 PM Treating Consultant: Dr. Rajesh Kumar, General_medicine Chief Complaints: Abdominal pain for 22 days. History of Present Illness: Productive cough, no hemoptysis. Past History: Diabetes on insulin. Physical Examination: Temp: 102°F; Chest: Crepitations. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: Endoscopy: Gastritis. Diagnosis: Community-Acquired Pneumonia (ICD: J18.9). Course in Hospital: IV antibiotics. Fever subsided. Socio-Economic Status: Class IV as per modified Kuppuswamy scale. Traditional Medicine History: Used Haldi milk for pain. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Diet: Bland; Follow-up: 1 week. Dr. Vikram Singh Signature"
4,"AIIMS Delhi Discharge Summary - V2 Patient Name: Imran Khan Age/Sex: 35/M UHID: 926990054 Admission No: ADM-20263390 Address: Darbhanga, Bihar Admission Date: 09/03/2026 11:00 AM Discharge Date: 13/03/2026 02:00 PM Treating Consultant: Dr. Arvind Subramanian (Cardiologist), Pediatrics Chief Complaints: Loose motions aur vomiting for 17 days. History of Present Illness: High fever, measles-like rash. Past History: Recent URI. Physical Examination: Dehydrated, abdomen soft. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: CRP: Positive. Diagnosis: Viral Exanthem (ICD: B09). Cours

In [0]:
display(
    df.filter(~F.col("note").contains("chief Complaints:"))
      .select("id", "note")
)

id,note
1,"SSKM Hospital Kolkata Discharge Summary - V2 Patient Name: Baby Saanvi Age/Sex: 68/F UHID: 632376936 Admission No: ADM-20260462 Address: Patna, Bihar Admission Date: 07/03/2026 08:00 AM Discharge Date: 14/03/2026 01:00 PM Treating Consultant: Dr. Naresh Trehan (Cardiac Surgeon), Oncology Chief Complaints: Unexplained weight loss and fatigue for 8 months. History of Present Illness: Patient noticed a gradual loss of appetite. Past History: No history of major illness. Physical Examination: 2x2 cm hard lump in the upper outer quadrant of the left breast. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: CBC shows anemia. PET scan shows metastasis. Diagnosis: Metastatic Adenocarcinoma of unknown primary (ICD: C80.0). Course in Hospital: Underwent mastectomy followed by chemotherapy. Socio-Economic Status: Class IV as per modified Kuppuswamy scale. Traditional Medicine History: Used Tulsi for cough. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Regular follow-up in the oncology OPD. - Dawa time pe lena. Dr. Naresh Trehan (Cardiac Surgeon) Signature"
2,"Tata Memorial Hospital Mumbai Discharge Summary - V2 Patient Name: Debashish Roy Age/Sex: 85/F UHID: 247712229 Admission No: ADM-20268090 Address: Kaithal, Haryana Admission Date: 09/03/2026 09:00 AM Discharge Date: 14/03/2026 02:00 PM Treating Consultant: Dr. Anita Sharma, Cardiology Chief Complaints: Palpitations and dizziness for 3 days. History of Present Illness: Substernal pain on exertion, with diaphoresis. Patient ko fever tha. Past History: Coronary artery disease, post-stent. Physical Examination: BP: 160/102 mmHg, Heart: S1/S2 normal, no murmur. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: ECG: ST elevation; Trop-I: Positive; Echo: EF 45%; Angiogram: 52% blockage in LAD. Diagnosis: Atrial Fibrillation (ICD: I48.91). Course in Hospital: Antiarrhythmics given, rhythm stabilized. Socio-Economic Status: Class III as per modified Kuppuswamy scale. Traditional Medicine History: Used Tulsi for cough. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Meds: Tab. Aspirin 75mg OD, Tab. Atorvastatin 40mg HS. - pura course complete karna. Dr. Arvind Subramanian (Cardiologist) Signature"
3,"Apollo Hospital Chennai Discharge Summary - V2 Patient Name: Jayesh Patel Age/Sex: 85/F UHID: 805408206 Admission No: ADM-20267292 Address: Kaithal, Haryana Admission Date: 03/03/2026 09:00 AM Discharge Date: 10/03/2026 03:00 PM Treating Consultant: Dr. Rajesh Kumar, General_medicine Chief Complaints: Abdominal pain for 22 days. History of Present Illness: Productive cough, no hemoptysis. Past History: Diabetes on insulin. Physical Examination: Temp: 102°F; Chest: Crepitations. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: Endoscopy: Gastritis. Diagnosis: Community-Acquired Pneumonia (ICD: J18.9). Course in Hospital: IV antibiotics. Fever subsided. Socio-Economic Status: Class IV as per modified Kuppuswamy scale. Traditional Medicine History: Used Haldi milk for pain. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Diet: Bland; Follow-up: 1 week. Dr. Vikram Singh Signature"
4,"AIIMS Delhi Discharge Summary - V2 Patient Name: Imran Khan Age/Sex: 35/M UHID: 926990054 Admission No: ADM-20263390 Address: Darbhanga, Bihar Admission Date: 09/03/2026 11:00 AM Discharge Date: 13/03/2026 02:00 PM Treating Consultant: Dr. Arvind Subramanian (Cardiologist), Pediatrics Chief Complaints: Loose motions aur vomiting for 17 days. History of Present Illness: High fever, measles-like rash. Past History: Recent URI. Physical Examination: Dehydrated, abdomen soft. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: CRP: Positive. Diagnosis: Viral Exanthem (ICD: B09). Cours

In [0]:
display(
    df.select("id", "specialty", "note")
      .limit(10)
)

id,specialty,note
1,oncology,"SSKM Hospital Kolkata Discharge Summary - V2 Patient Name: Baby Saanvi Age/Sex: 68/F UHID: 632376936 Admission No: ADM-20260462 Address: Patna, Bihar Admission Date: 07/03/2026 08:00 AM Discharge Date: 14/03/2026 01:00 PM Treating Consultant: Dr. Naresh Trehan (Cardiac Surgeon), Oncology Chief Complaints: Unexplained weight loss and fatigue for 8 months. History of Present Illness: Patient noticed a gradual loss of appetite. Past History: No history of major illness. Physical Examination: 2x2 cm hard lump in the upper outer quadrant of the left breast. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: CBC shows anemia. PET scan shows metastasis. Diagnosis: Metastatic Adenocarcinoma of unknown primary (ICD: C80.0). Course in Hospital: Underwent mastectomy followed by chemotherapy. Socio-Economic Status: Class IV as per modified Kuppuswamy scale. Traditional Medicine History: Used Tulsi for cough. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Regular follow-up in the oncology OPD. - Dawa time pe lena. Dr. Naresh Trehan (Cardiac Surgeon) Signature"
2,cardiology,"Tata Memorial Hospital Mumbai Discharge Summary - V2 Patient Name: Debashish Roy Age/Sex: 85/F UHID: 247712229 Admission No: ADM-20268090 Address: Kaithal, Haryana Admission Date: 09/03/2026 09:00 AM Discharge Date: 14/03/2026 02:00 PM Treating Consultant: Dr. Anita Sharma, Cardiology Chief Complaints: Palpitations and dizziness for 3 days. History of Present Illness: Substernal pain on exertion, with diaphoresis. Patient ko fever tha. Past History: Coronary artery disease, post-stent. Physical Examination: BP: 160/102 mmHg, Heart: S1/S2 normal, no murmur. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: ECG: ST elevation; Trop-I: Positive; Echo: EF 45%; Angiogram: 52% blockage in LAD. Diagnosis: Atrial Fibrillation (ICD: I48.91). Course in Hospital: Antiarrhythmics given, rhythm stabilized. Socio-Economic Status: Class III as per modified Kuppuswamy scale. Traditional Medicine History: Used Tulsi for cough. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Meds: Tab. Aspirin 75mg OD, Tab. Atorvastatin 40mg HS. - pura course complete karna. Dr. Arvind Subramanian (Cardiologist) Signature"
3,general_medicine,"Apollo Hospital Chennai Discharge Summary - V2 Patient Name: Jayesh Patel Age/Sex: 85/F UHID: 805408206 Admission No: ADM-20267292 Address: Kaithal, Haryana Admission Date: 03/03/2026 09:00 AM Discharge Date: 10/03/2026 03:00 PM Treating Consultant: Dr. Rajesh Kumar, General_medicine Chief Complaints: Abdominal pain for 22 days. History of Present Illness: Productive cough, no hemoptysis. Past History: Diabetes on insulin. Physical Examination: Temp: 102°F; Chest: Crepitations. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: Endoscopy: Gastritis. Diagnosis: Community-Acquired Pneumonia (ICD: J18.9). Course in Hospital: IV antibiotics. Fever subsided. Socio-Economic Status: Class IV as per modified Kuppuswamy scale. Traditional Medicine History: Used Haldi milk for pain. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Diet: Bland; Follow-up: 1 week. Dr. Vikram Singh Signature"
4,pediatrics,"AIIMS Delhi Discharge Summary - V2 Patient Name: Imran Khan Age/Sex: 35/M UHID: 926990054 Admission No: ADM-20263390 Address: Darbhanga, Bihar Admission Date: 09/03/2026 11:00 AM Discharge Date: 13/03/2026 02:00 PM Treating Consultant: Dr. Arvind Subramanian (Cardiologist), Pediatrics Chief Complaints: Loose motions aur vomiting for 17 days. History of Present Illness: High fever, measles-like rash. Past History: Recent URI. Physical Examination: Dehydrated, abdomen soft. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: 

In [0]:
display(
    df.filter(F.col("specialty") == "cardiology")
      .select("id", "note")
      .limit(2)
)

id,note
2,"Tata Memorial Hospital Mumbai Discharge Summary - V2 Patient Name: Debashish Roy Age/Sex: 85/F UHID: 247712229 Admission No: ADM-20268090 Address: Kaithal, Haryana Admission Date: 09/03/2026 09:00 AM Discharge Date: 14/03/2026 02:00 PM Treating Consultant: Dr. Anita Sharma, Cardiology Chief Complaints: Palpitations and dizziness for 3 days. History of Present Illness: Substernal pain on exertion, with diaphoresis. Patient ko fever tha. Past History: Coronary artery disease, post-stent. Physical Examination: BP: 160/102 mmHg, Heart: S1/S2 normal, no murmur. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: ECG: ST elevation; Trop-I: Positive; Echo: EF 45%; Angiogram: 52% blockage in LAD. Diagnosis: Atrial Fibrillation (ICD: I48.91). Course in Hospital: Antiarrhythmics given, rhythm stabilized. Socio-Economic Status: Class III as per modified Kuppuswamy scale. Traditional Medicine History: Used Tulsi for cough. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Meds: Tab. Aspirin 75mg OD, Tab. Atorvastatin 40mg HS. - pura course complete karna. Dr. Arvind Subramanian (Cardiologist) Signature"
10,"Fortis Hospital Delhi Discharge Summary - V2 Patient Name: Ananya Banerjee Age/Sex: 8 months/M UHID: 585896652 Admission No: ADM-20262756 Address: Connaught Place, New Delhi Admission Date: 02/03/2026 10:00 AM Discharge Date: 15/03/2026 01:00 PM Treating Consultant: Dr. Vikram Singh, Cardiology Chief Complaints: Palpitations and dizziness for 26 days. History of Present Illness: Irregular heartbeat, no syncope. Past History: Coronary artery disease, post-stent. Physical Examination: BP: 175/100 mmHg, Heart: S1/S2 normal, no murmur. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: ECG: ST elevation; Trop-I: Positive; Echo: EF 51%; Angiogram: 79% blockage in LAD. Diagnosis: NSTEMI (ICD: I21.4). Course in Hospital: Antiarrhythmics given, rhythm stabilized. Socio-Economic Status: Class II as per modified Kuppuswamy scale. Traditional Medicine History: Used Haldi milk for pain. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Meds: Tab. Aspirin 75mg OD, Tab. Atorvastatin 40mg HS. Dr. S. H. Advani (Oncologist) Signature"


In [0]:
display(
    df.filter(F.col("specialty") == "neurology")
      .select("id", "note")
      .limit(2)
)

id,note
8,"AIIMS Delhi Discharge Summary - V2 Patient Name: Dinesh Kulkarni Age/Sex: Newborn/M UHID: 560010357 Admission No: ADM-20264932 Address: Bandra West, Mumbai Admission Date: 02/03/2026 08:00 AM Discharge Date: 12/03/2026 02:00 PM Treating Consultant: Dr. Alok Sharma (Neurosurgeon), Neurology Chief Complaints: Right side weakness aur bolne mein difficulty for 23 days. History of Present Illness: Migraine-like, photophobia. saans lene mein takleef thi. Past History: Hypertension, smoker. Physical Examination: Power: 3/5 right limbs; Reflexes: Hyperactive. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: MRI: Left MCA infarct; CT: No bleed. Diagnosis: Ischemic Stroke (ICD: I63.9). Course in Hospital: IV thrombolysis, physiotherapy. Improved. Socio-Economic Status: Class IV as per modified Kuppuswamy scale. Traditional Medicine History: Used Haldi milk for pain. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Avoid triggers; Follow-up: 10 days. - pura course complete karna. Dr. S. H. Advani (Oncologist) Signature"
11,"SSKM Hospital Kolkata Discharge Summary - V2 Patient Name: Mohammed Ali Age/Sex: 72/M UHID: 964717599 Admission No: ADM-20269925 Address: Bandra West, Mumbai Admission Date: 06/03/2026 08:00 AM Discharge Date: 10/03/2026 02:00 PM Treating Consultant: Dr. Vikram Singh, Neurology Chief Complaints: Severe headache with nausea for 20 days. History of Present Illness: Sudden onset, with facial droop. Past History: Hypertension, smoker. Physical Examination: Power: 3/5 right limbs; Reflexes: Hyperactive. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: - Investigations: MRI: Left MCA infarct; CT: No bleed. Diagnosis: Migraine (ICD: G43.9). Course in Hospital: IV thrombolysis, physiotherapy. Improved. Socio-Economic Status: Class IV as per modified Kuppuswamy scale. Traditional Medicine History: Used Haldi milk for pain. Condition at Discharge: Stable, afebrile. Advice on Discharge: - Meds: Tab. Clopidogrel 75mg OD. - Dawa time pe lena. Dr. Anjali Desai (Pediatrician) Signature"
